In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models
import pickle

: 

In [ ]:
data_dir = "../data"
x_list = []
y_list = []
batch_size = 32

In [ ]:
for file in os.listdir(data_dir):
    path = os.path.join(data_dir, file)
    with open(path, 'rb') as fo:
        data_dict = pickle.load(fo, encoding='bytes')

        x_list.append(data_dict[b'data'])
        y_list.extend(data_dict[b'labels'])

In [ ]:
X = np.concatenate(x_list)
Y = np.array(y_list)

In [ ]:
X = X.reshape(-1, 3,32,32)
X = X.transpose(0,2,3,1)
X = X.astype(float) / 255.0

In [ ]:
y = to_categorical(Y, 10)

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices((X, y))

dataset = dataset.shuffle(buffer_size=len(X), seed=42)

In [ ]:
train_size = int(0.8*len(X))

In [ ]:
train_dataset = dataset.take(train_size)
val_dataset = dataset.skip(train_size)

In [ ]:
train_dataset = train_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
print(len(X), X.shape)
print(len(y), y.shape if isinstance(y, np.ndarray) else 'no es ndarray')

In [ ]:
IMG_SIZE = (32,32) + (3,)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE, include_top=False, weights='imagenet')

In [ ]:
base_model.summary()

# Congelación de capas

In [ ]:
base_model.trainable = True

In [ ]:
for layer in base_model.layers[:50]:
    layer.trainable = False

In [ ]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x= layers.Dense(128, activation='relu')(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dense(32, activation='relu')(x)
output = layers.Dense(10, activation='softmax')(x)

In [ ]:
model = models.Model(inputs=base_model.input, outputs=output)

In [ ]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_dataset, 
    validation_data=val_dataset,
    epochs=10

)